# BT4012: Bitcoin Transaction Fraud Detection

**Student number:** E1156285

This is the starter notebook for the individual Kaggle competition. It currently loads and checks the data. We will add preprocessing, feature engineering, chronological validation, model comparison, and final submission generation as the assignment progresses.

## How to use this notebook

- A **Markdown cell** contains explanations.
- A **Code cell** runs Python. Click a cell and press **Shift + Enter** to run it and move to the next cell.
- Run cells from top to bottom. Variables remain in the running Python session, called the **kernel**.
- Use **Kernel > Restart Kernel and Run All Cells** to check that everything works from a fresh session.
- Save with **Command + S** on macOS.

Keep your original CSV files unchanged. Blank labels are unknown, not legitimate.


## 1. Import libraries and locate the data

`Path` helps locate files. `pandas` reads CSV files into tables called DataFrames. `numpy` provides numerical operations.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data Set"
assert DATA_DIR.is_dir(), "Start Jupyter from the Kaggle project folder containing Data Set."
print("Data folder:", DATA_DIR)


Data folder: /Users/mlvlo/BT4012/Kaggle/Data Set


## 2. Load the four files

Training contains earlier transactions and some labels. Test contains later transactions requiring predictions. Edges describe payment dependencies. The submission template defines the required index and target columns.


In [2]:
train = pd.read_csv(DATA_DIR / "Train Data.csv")
test = pd.read_csv(DATA_DIR / "Test Data.csv")
edges = pd.read_csv(DATA_DIR / "TXS Edgelist.csv")
sample_submission = pd.read_csv(DATA_DIR / "Sample Submission" / "BT4012 Competition Submission.csv")

for name, table in [("Training", train), ("Test", test), ("Edges", edges), ("Submission template", sample_submission)]:
    print(f"{name}: {table.shape[0]:,} rows, {table.shape[1]} columns")


Training: 141,772 rows, 168 columns
Test: 15,329 rows, 168 columns
Edges: 234,355 rows, 2 columns
Submission template: 15,329 rows, 2 columns


## 3. View a few transactions

Each training row represents a transaction. `NaN` in label means its answer is unknown. Feature names are anonymized, so we do not assign invented meanings to them.


In [3]:
train[["txId", "time_step", "feat_1", "feat_2", "label"]].head()


,txId,time_step,feat_1,feat_2,label
0,230425980,1,-0.171469,-0.184668,NaN
1,5530458,1,-0.171484,-0.184668,NaN
2,232022460,1,-0.172107,-0.184668,NaN
3,232438397,1,0.163054,1.963790,0.0
4,230460314,1,1.011523,-0.081127,NaN


## 4. Separate known answers from unknown answers

`1` means illicit. `0` means licit. Only known labels are used as supervised training targets. This cell prepares separate tables; it does not train a model.


In [4]:
labels = pd.to_numeric(train["label"], errors="coerce")
assert labels.dropna().isin([0, 1]).all(), "Unexpected label values."
labeled = train.loc[labels.notna()].copy()
labeled["label"] = labels.loc[labels.notna()].astype(int)
unlabeled = train.loc[labels.isna()].copy()

print("Licit:", int((labeled["label"] == 0).sum()))
print("Illicit:", int((labeled["label"] == 1).sum()))
print("Unknown:", len(unlabeled))
print(f"Illicit share among known labels: {labeled['label'].mean():.2%}")


Licit: 27591
Illicit: 3644
Unknown: 110537
Illicit share among known labels: 11.67%


## 5. Check the features, timeline, and submission IDs

The final evaluation is on later periods. Our validation will therefore use earlier periods to train and later periods to check predictions. `txId` joins graph connections; `index` identifies submission rows.


In [5]:
FEATURE_COLUMNS = [f"feat_{i}" for i in range(1, 166)]
assert all(column in train.columns and column in test.columns for column in FEATURE_COLUMNS)
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(set(test["txId"]))
assert train["time_step"].max() < test["time_step"].min()
assert sample_submission["index"].equals(test["index"])
assert not train[FEATURE_COLUMNS].isna().any().any()
assert not test[FEATURE_COLUMNS].isna().any().any()
assert np.isfinite(train[FEATURE_COLUMNS].to_numpy()).all()
assert np.isfinite(test[FEATURE_COLUMNS].to_numpy()).all()

print(f"Training time steps: {train.time_step.min()}–{train.time_step.max()}")
print(f"Test time steps: {test.time_step.min()}–{test.time_step.max()}")
print("Feature columns:", len(FEATURE_COLUMNS))
print("Basic checks passed. No model has been trained yet.")


Training time steps: 1–35
Test time steps: 36–49
Feature columns: 165
Basic checks passed. No model has been trained yet.


## 6. Work to complete

1. Explore feature distributions and label patterns over time.
2. Design chronological validation.
3. Train and evaluate a simple baseline.
4. Compare additional models and graph features.
5. Record actual results and justify the final model.
6. Train the selected model and generate a CSV with `index,target`.

The template contains 0.5 placeholders. Do not present it as a trained model submission.

## Assistance acknowledgement

OpenAI Codex assisted with repository setup and this starter notebook. Update this acknowledgement to accurately describe assistance used throughout the project, and ensure you can explain all submitted work.
